# Silver Exploration Notebook


## Imports

In [ ]:
import os
import findspark

import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.sql.functions import (
    col,
    lit,
    when,
    to_timestamp,
    from_utc_timestamp,
    dayofweek,
    hour,
    month,
    input_file_name,
    regexp_extract,
    arrays_zip,
    explode,
    size,
    count,
    length,
    min,
    max,
    avg,
    slice,
    regexp_extract,
    desc,
    unix_micros,
    abs,
    split,
    element_at,
    countDistinct,
    percentile_approx,
    radians,
    sin,
    cos,
    asin,
    sqrt,
    pow,
    when,
    floor,
    sum as spark_sum
)

## Start Spark

In [ ]:
findspark.init()

spark = (
    SparkSession.builder
    .appName("silver-exploration")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

In [ ]:
RAW = os.path.expanduser("~/spark-lab-data/dev/project/raw")
print(RAW)

## Read Bike-sharing dataset
+ Adding city and month

In [ ]:
# Bike-sharing data
trips_raw = (
    spark.read
    .option("header", True)
    .csv(f"{RAW}/trips/*/*.csv")
    .withColumn("src", input_file_name())
)

trips_raw = (
    trips_raw
    .withColumn("city", element_at(split(col("src"), "/"), -2))
    .withColumn("year_month", element_at(split(col("src"), "/"), -1))
    .withColumn("year_month",split(col("year_month"), r"\.").getItem(0))
)

trips_raw.select("city", "year_month").show(5)

## 1. Exploration of Bike-sharing dataset

In [ ]:
trips_raw.printSchema()
trips_raw.show(5, truncate=False)
print("Total rows:", trips_raw.count())
print("Columns:", trips_raw.columns)

- We have more than 8 470 000 bike trips in Oslo, Trondheim and Bergen.
- The columns in the Bike-sharing dataset include: `started_at`, `ended_at`, `duration`, `start_station_id`, `start_station_name`, `start_station_description`, `start_station_latitude`, `start_station_longitude`, `end_station_id`, `end_station_name`, `end_station_description`, `end_station_latitude`, `end_station_longitude`. We have also added `src`, `city` and `year_month`.

#### Data Types
The 16 columns are stored as strings. This is because PySpark reads CSV columns as strings by default. However, several columns represent numeric values or timestamps. We will therefore convert these columns to appropriate data types. 

- Timestamps: `started_at`and `ended_at` -> timestamp
- Trip duration: `duration` -> double
- Coordinates: `latitude`and `longitude`-> double

#### What about the stations IDs?
- Station IDs: `start_station_id` and `stop_station_id` -> keep as strings or convert to integers?

From the table above we can see the ID is stored as a numeric value. However, we would argue to keep them as a string, as they are identifiers rather than numeric measurements. This also provides greater flexibility if future datasets contain letters(e.g. ST123), without requiring changes to the data type. 


In [ ]:
trips_silver = trips_raw

## 2. Converting data types

### duration: string -> double

In [ ]:
trips_silver = (
    trips_silver
    .withColumn("duration", col("duration").cast("double"))
)

# Did the convertion work?
trips_silver.filter(
    col("duration").isNull() 
).count()

### started_at / ended_at: string -> timestamp
The `started_at` and `ended_at` timestamps are recorded in UTC, as indicated by `+00:00`.

For example:
- `started_at`: 2022-08-01 05:14:13.781000+00:00
- `ended_at`: 2022-08-01 05:26:27.585000+00:00

In [ ]:
# Setting the spark session timezone to UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Converting the started_at and ended_at columns to timestamp type
trips_silver = (
    trips_silver
    .withColumn("started_ts", to_timestamp(col("started_at")))
    .withColumn("ended_ts", to_timestamp(col("ended_at")))
)

# Let's compare the converted timestamps
trips_silver.select(
    "started_at",
    "started_ts",
    "ended_at",
    "ended_ts"
).show(10, truncate=False)

# Did the convertion work?
trips_silver.filter(
    col("started_ts").isNull() |
    col("ended_ts").isNull()
).count()

### latitude/ longitude: string -> double

In [ ]:
trips_silver = (
    trips_silver
    .withColumn("start_station_latitude", col("start_station_latitude").cast("double"))
    .withColumn("start_station_longitude", col("start_station_longitude").cast("double"))
    .withColumn("end_station_latitude", col("end_station_latitude").cast("double"))
    .withColumn("end_station_longitude", col("end_station_longitude").cast("double"))
)

# Did the convertion work?
trips_silver.filter(
    col("start_station_latitude").isNull() |
    col("start_station_longitude").isNull() |
    col("end_station_latitude").isNull() |
    col("end_station_longitude").isNull()
).count()


## 3. Missing values and Duplicates

In [ ]:
trips_silver.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in trips_silver.columns
]).show()

#### Missing values
The `start_station_description` column contains around 11 000 missing values, while `end_station_description` contains around 7000 missing values.

#### Is this detrimental to our project or analysis?
These columns are not truly important for the type of analysis we want to do. Our main objective is to investigate how weather conditions influence bike-sharing usage. Since station IDs and names are available, the missing descriptions do not prevent us from identifying stations and analyzing trips. 

Although they are not required for our current objectives, they could be valuable for other purposes later. In the Gold layer, we can simply exclude columns that are not relevant to our final analytical dataset. 

### Investigating duplicates

In [ ]:
print("Total rows:", trips_silver.count())
print("Duplicate rows:", trips_silver.count() - trips_silver.distinct().count())

From more than 8 472 000 trips, 25 additional duplicates. 

#### Are we sure these are duplicates?

We cannot be certain that these are duplicates because the trips do not contain any identifier. These records do however contain identical entries across all 16 columns, including timestamp with microsecond precision. It is highly unlikely that separate trips would share all these values.

We therefore consider these records to be duplicate entries, potentially from the source data or overlapping records during the ingestion process. The exact cause has not been investigated further, but we want to remove the additional copies. 

Given that our dataset contains over 8 million trips, removing 25 duplicate records will have a negligible impact on the overall analysis. If the dataset were significantly smaller, further investigation would be important as removing even a small number of records could have a greater influence on the result.

In [ ]:
rows_in = trips_raw.count()

duplicate_cols = [
    c for c in trips_silver.columns
    if c not in ["src", "year_month"]
]

trips_silver = trips_silver.dropDuplicates(duplicate_cols)

rows_out = trips_silver.count()

duplicated_rows = rows_in - rows_out

print(f"Total rows before: {rows_in}")
print(f"Total rows after: {rows_out}")
print(f"Duplicates removed: {rows_in} - {rows_out} = {duplicated_rows}")

### Why not flag the duplicates?
Unlike potentially unrealistic data, duplicate records may lead to double-counting when aggregating bike usage. We record removed duplicates as a quality metric to maintain transparency. 

## 4. Timestamp Validation

### Investigating the timestamp format

In [ ]:
trips_silver.groupBy(length("started_at").alias("len")).count().orderBy("len").show()
trips_silver.select("started_ts", "ended_ts", "duration").distinct().limit(10).show(truncate=False)

### Converting Timestamps to Norwegian Local Time

We convert these timestamps to Norwegian local time (`Europe/Oslo`), accounting for daylight saving time. The original UTC timestamps are preserved for further processing.

In [ ]:
trips_silver = (
    trips_silver
    .withColumn(
        "started_local",
        from_utc_timestamp("started_ts", "Europe/Oslo")
    )
    .withColumn(
        "ended_local",
        from_utc_timestamp("ended_ts", "Europe/Oslo")
    )
    
)

trips_silver.select("started_ts","started_local","ended_ts","ended_local").show(10, truncate=False)

#### Does duration correspond to the start and end time?

In [ ]:
trips_silver.select(
    "started_ts",
    "ended_ts",
    "duration",
    # does started_ts and ended_ts match the duration?
    (
        (
            unix_micros("ended_ts") -
            unix_micros("started_ts")
        ) / 1000000
    ).alias("duration_check")

).show(5, truncate=False)

From the table above it looks like the duration is always rounded down.

for example:
- 363 and 363.712
- 703 and 703
- 778 and 778.756

Let's investigate the duration a bit further.

In [ ]:
trips_check = trips_silver.withColumn(
    "duration_diff",
    abs(
        (unix_micros("ended_ts")
         - unix_micros("started_ts")) / 1_000_000
        - col("duration").cast("double")
    )
)

trips_check.select("duration_diff").summary().show()

Since the maximum difference is below one second, we consider the recorded durations consistent with the timestamps. 

## 5. Coordinate Validation

#### Valid coordination for latitude and longitude in Norway:
- Norwegian latitude: 57 - 71
- Norwegian longitude: 4 - 31

In [ ]:
trips_silver.filter(
    (col("start_station_latitude") < 57) |
    (col("start_station_latitude") > 71) |
    (col("end_station_latitude") < 57) |
    (col("end_station_latitude") > 71) |
    (col("start_station_longitude") < 4) |
    (col("start_station_longitude") > 31) |
    (col("end_station_longitude") < 4) |
    (col("end_station_longitude") > 31)
).count()

In [ ]:
trips_silver.select(
    min("start_station_latitude").alias("min_lat"),
    max("start_station_latitude").alias("max_lat"),
    min("start_station_longitude").alias("min_lon"),
    max("start_station_longitude").alias("max_lon")
).show()

The latitude and longitude values appear to be reasonable based on the geographical locations of the three cities. Oslo is the southernmost city, while Trondheim is the northernmost city. Bergen is the city most western, while oslo is the city most eastern. 

|   City       |   latitude    |   longitude   |
|--------------|---------------|---------------|
|  Oslo:       |     59.91     |     10.75     |  
|  Bergen:     |     60.39     |      5.32     |
|  Trondheim:  |     63.43     |     10.40     |

## 6. Trip Duration and Distance Analysis

### Trip duration distribution

In [ ]:
trips_silver.select("duration").describe().show()

In [ ]:
trips_silver.select("duration").summary("min", "25%", "50%", "75%", "max").show()

trips_silver.select(spark_sum(when(col("duration") <= 0, 1).otherwise(0)).alias("non_positive"),
         spark_sum(when(col("duration") < 60, 1).otherwise(0)).alias("under_60s"),
         spark_sum(when(col("duration").isNull(), 1).otherwise(0)).alias("null_duration")).show()

#### Durations to investigate further
- 2 trips have a negative or a duration of 0.
- Is it possible that a trip lasts around 22 hours? (79790/3600 = 22)

In [ ]:
# Trips with durations smaller than 1 second
trips_silver.filter(
    col("duration").cast("double") <= 1
).show(truncate=False)

print("How many trips have a duration of 1 second or less?", trips_silver.filter(col("duration").cast("double") <= 1).count())

### Extremely Short Trips

29 trips with a recorded duration of one second or less, including two trips with a duration of zero seconds. 

With the table above these trips have a different start and end station which makes it physically impossible to complete the journeys within that timeframe.

These records likely represent data quality issues that need further investigation.

Possible solution: 
- Find the mean/median trip duration from one station to the other and compare it to minimum duration record.
- Flag trips under a specific timeframe. 
- Anything else?

In [ ]:
station_durations = (
    trips_silver
    .groupBy("city", "start_station_id", "end_station_id")
    .agg(
        count("*").alias("num_trips"),
        avg("duration").alias("mean_duration"),
        min("duration").alias("min_duration"),
        percentile_approx("duration", 0.5).alias("median_duration")
    )
)

station_durations.filter(
    col("num_trips") >= 10
).orderBy("min_duration").show(20, truncate=False)

#### What does the table above tell us?
Although the mean and median durations do not actually tell us the minimum possible duration between two stations, they provides a useful indication of what a typical trip looks like. 

The table shows station pairs with hundreds and thousands of bike-trips, where the average duration is several minutes, yet the shortest recorded time is only zero or one second. This strongly suggests that some values are incorrect or unrealistic. 

We also see that the mean duration is constantly higher than the median for the station pairs, suggesting that long trips may influence the mean. The median may therefore provide a more robust reference for identifying unusually short trips.

On another note this does make me question the dataset and its accuracy. Are these simply a small number of isolated data quality issues, or could they indicate more problems with the data?

For now let's at least flag trips with a duration smaller than one second. Further investigation is needed to determine how many trips have a duration significantly below the median of their station pairs and whether these records could affect downstream analyses. 

In [ ]:
# Flag trips that are under 1 second
trips_silver = trips_silver.withColumn(
    "short_trip_flag",
    col("duration") <= 1
)

### Station pair duration analysis

In [ ]:
total_trips = trips_silver.count()

trips_silver = trips_silver.join(
    station_durations.select(
        "city",
        "start_station_id",
        "end_station_id",
        "median_duration",
        "num_trips"
    ),
    on = ["city", "start_station_id", "end_station_id"],
    how = "left"
)

for threshold in [0.01, 0.05, 0.10, 0.20]:
    short_trips = trips_silver.filter(
        (col("num_trips") >= 10) &
        (col("median_duration") > 0) &
        (col("duration") < col("median_duration") * threshold)
    ).count()

    percentage = (short_trips / total_trips) * 100

    print(f"Below {threshold:.0%} of median: " 
          f"{short_trips} trips ({percentage:.6f})")

The results: 
- Below 1% of median: 56 trips (0.000661)
- Below 5% of median: 1230 trips (0.014519)
- Below 10% of median: 10995 trips (0.129785)
- Below 20% of median: 36769 trips (0.434021)

This investigated only station pairs with a minimum of ten recorded trips and a positive median duration.

Since a short trip duration relative to the median does not necessarily indicate an invalid trip, introducing a higher threshold could incorrectly flag valid trips. 

To investigate this further, we can find the straight-line distance between the stations pairs and use the recorded duration to estimate the minimum required average speed. 

### Calculate straight-line distance

In [ ]:
# Converting coordinates from degrees to radians
lat1 = radians(col("start_station_latitude"))
lat2 = radians(col("end_station_latitude"))

lon1 = radians(col("start_station_longitude"))
lon2 = radians(col("end_station_longitude"))

# Haversine formula
a = (
    pow(sin((lat2 - lat1) / 2), 2) +
    cos(lat1) * cos(lat2) *
    pow(sin((lon2 - lon1) / 2), 2)
)

# Calulate straight-lin distance in kilometers
trips_silver = trips_silver.withColumn(
    "distance_km",
     2 * 6371 * asin(sqrt(a))
)


trips_silver.select(
    "city",
    "start_station_id",
    "end_station_id",
    "duration",
    "median_duration",
    "distance_km"
).show(20, truncate=False)

This estimate only calculate the minimum kilometers traveled, and is not a true representation of what distance is actually traveled. However, this can help be a guideline to find unrealistic data. 

### Minumum implied average speed

In [ ]:
trips_silver = trips_silver.withColumn(
    "speed_kmh",
    when(
        (col("duration") > 0) &
        (col("distance_km").isNotNull()),
        (col("distance_km") * 3600) / col("duration")
    )
)

trips_silver.select(
    "city",
    "start_station_id",
    "end_station_id",
    "duration",
    "median_duration",
    "distance_km",
    "speed_kmh"
).orderBy(
    col("speed_kmh").desc()
).show(20, truncate=False)

### Unrealistic cycling speeds 

According to vegvesenet, electric bicycles must stop providing motor assistance when reaching a speed of 25 km/h. You may exceed this speed through pedalling or downhill cycling. A speed over 100 km/h is highly unlikely. A speed of around 12 000 km/h must be some type of record, as it's more than twice of the speed of most planes. 

Since our calculated speed is based on the straight-line distance between stations, it therefore represents a lower bound for the actual cycling speed. To investigate potential unrealistic trips, we can evaluate different speed thresholds.

In [ ]:
for threshold in [25, 30, 40, 50, 60, 100]:
    fast_trips = trips_silver.filter(
        col("speed_kmh") > threshold
    ).count()

    percentage_speed = (fast_trips / total_trips) * 100

    print(f"Above {threshold} km/h "
          f"{fast_trips} trips ({percentage_speed:.6f}%)")

The results show that 1786 trips have a mimimum average speed exceeding 25 km/h. More notably 874 trips exceed 60 km/h and 567 trips exceed 100 km/h.

Even though these records represent a small proportion of the dataset, the extremely high estimated speeds strongly suggest data quality issues. 

Compared to the median-based approach, the stright-line distance method provides greater certanty that the trips are in fact unrealistic. 

We could therefore introduce an additional speed-based quality flag to improve the reliability. 

In [ ]:
additional_invalid_trips = trips_silver.filter(
    (col("speed_kmh") > 60) &
    (col("duration") <= 1)
).count()

print(f"Trips that are both a speed over 60 km/h and 1 secound: " f"{additional_invalid_trips}")

We know that there are 29 trips with a duration with one second or less. We also know that there are 874 trips with a speed over 60 km/h and now we know that 27 of those trips around 1 second are included in the trips that are over a speed of 60 km/h. 

We would therefore conclude with that we can have both instances, both flag trips around one second and flag trips with a speed over 60 km/h. 

In [ ]:
# Flag trips that have a speed over 60 km/h
trips_silver = trips_silver.withColumn(
    "unrealistic_speed_flag",
    col("speed_kmh") > 60
)

### Trips with a long duration

In [ ]:
print("Number of trips longer than 12 hours: ", trips_silver.filter(col("duration") > 12 * 3600).count())

trips_silver.select(
    "started_at",
    "ended_at",
    "duration"
).orderBy(desc("duration")).show(15, truncate=False)

Could a city bike realistically be used for appoximately 22 hours?

According to Oslo Bysykkel, a fully charged electric bike has an estimated range of around 70 kilometers. The battery does not necessarily limit the recorded trip duration, as the bike can remain parked while the rental is still active and you can manually ride the bike if the battery runs out. 

If the dataset included distance travelled we could better investigate whether the bicycle was actively used.

Possible explanations for the long duration: 
- The user forgot to return the bike or didn't properly complete the return process.
- The bike was parked for several hours while the rental was still active. 
- The recorded timestamp contains an error. 
- Maybe others?

In [ ]:
print("Over 3 hours:", trips_silver.filter(col("duration") > 3 * 60 * 60).count())
print("Over 6 hours:", trips_silver.filter(col("duration") > 6 * 60 * 60).count())
print("Over 12 hours:", trips_silver.filter(col("duration") > 12 * 60 * 60).count())

#### Percentage of trips over 3 hours 

In [ ]:
total_trips = trips_silver.count()

long_trips = trips_silver.filter(
    col("duration") > 3 * 3600 
).count()

print(f"{long_trips / total_trips * 100}%")

(3184/8470330) * 100 = 0.0376 %

Only 0.038% of all bike trips last longer than 3 hours, indicating such trips are extremely rare.

### Round trips

Since some of the trips start and end at the same station, the straight-line distance for each of these trips will be zero. We would therefor like to investigate how common these trips are and whether they differ from other trips.

In [ ]:
trips_silver = trips_silver.withColumn(
    "round_trip_flag",
    F.col("start_station_id") == F.col("end_station_id")
)

#### Number of round trips per city

In [ ]:
trips_silver.groupBy("city").agg(
    F.count("*").alias("total_trips"),
    F.sum(
        F.col("round_trip_flag").cast("int")
    ).alias("round_trips"),
    (F.avg(F.col("round_trip_flag").cast("int")) * 100 ).alias("round_trip_percentage")
).show()

What is the different duration between round trips and other trips?

In [ ]:
trips_silver.groupBy(
    "city", "round_trip_flag"
).agg(
    F.count("*").alias("num_trips"),
    F.avg("duration").alias("mean_duration"),
    F.percentile_approx(
        "duration", 0.5
    ).alias("median_duration")
).show()

### Trip distance distribution

How does the typical straight-line distance between stations differ across cities?

In [ ]:
trips_silver.filter(
    ~F.col("round_trip_flag")
).groupBy("city").agg(
    F.avg("distance_km").alias("mean_distance"),
    F.percentile_approx("distance_km", 0.5).alias("median_distance"),
    F.max("distance_km").alias("max_distance")
).show()

Something else that would be interesting to investigate is whether the popularity and directions of these trips change throughout the day. We can first investigate which station pairs is most used per city. 

In [ ]:
routes = (
    trips_silver
    .filter(~F.col("round_trip_flag"))
    .groupBy(
        "city", 
        "start_station_id",
        "end_station_id"
    ).count()
)

window = Window.partitionBy("city").orderBy(
    F.desc("count"),
    "start_station_id",
    "end_station_id"
)

routes.withColumn(
    "rank", F.row_number().over(window)
).filter(
    F.col("rank") <= 10
).show(30)

In [ ]:
trips_silver = trips_silver.withColumn(
    "day_part",
    F.when(
        F.col("local_hour").between(7, 9),
        "morning_rush"
    ).when(
        F.col("local_hour").between(15, 17),
        "evening_rush"
    ).otherwise("other")
)

In [ ]:
trips_silver.filter(
    (~F.col("is_weekend")) &
    (~F.col("round_trip_flag"))
).groupBy(
    "city",
    "day_part",
    "start_station_id",
    "end_station_id"
).count().orderBy(
    F.desc("count")
).show(30)

## 7. Temporal Usage Patterns

### Trip distribution across cities

In [ ]:
trips_silver.groupBy("city").count().show()

Oslo accounts for approximately 68% of all trips in the dataset, so the analysis of the trips may be dominated by the travel patterns in Oslo.

#### Number of trips divided by city and stations

In [ ]:
trips_silver.groupBy("city", "start_station_id", "end_station_id").count().show(200)

When we observe the table above we see that:
- Some trip-pair have a significantly higher number of trips than others.
- Some trips start and end at the same station. 

### Monthly trip distribution

In [ ]:
trips_silver.groupBy(
    "city", "year_month"
).count().orderBy(
    "city", "year_month"
).show(200)

We know that Trondheim closes there bike-sharing in the winter when the snow comes. 

Further we want to investigate when people actually use the bikes throughout the day. Are there certain hours when the bike-sharing is more popular, and do these pattern differ between weekdays and weekends?

### Hourly trips distribution

In [ ]:
trips_silver = (
    trips_silver
    .withColumn("local_hour", F.hour("started_local"))
    .withCOlumn("weekday", F.dayofweek("started_local"))
    .withColumn(
        "is_weekend", F.col("weekday").isin(1, 7)
    )
)

#### Trips per hour

In [ ]:
trips_silver.groupBy(
    "city", "local_hour"
).count().orderBy(
    "city", "local_hour"
).show(72)

### Operating hours

In [ ]:
trips_silver.filter(
    F.col("local_hour").between(0, 6)
).groupBy(
    "city", "local_hour"
).count().orderBy(
    "city", "local_hour"
).show()

We can observe two distinxt peaks, one around 08:00 and anthoer around 15:00-16:00. This suggesting that a considerable proportion of the trips may be related to commuting. 

The other thing that caught my attention is the difference in bike-sharing usage during the night. Trondheim has no recorded trips between 03:00 and 06:00. The different cities do have different operating hours.  

Opning hours:
- Oslo:           05:00 - 01:00
- Trondheim:      06:00 - 00:00
- Bergen:         05:00 - 00:00

You can only rent a bike in the opening hours, but you can deliver the bike 24/7.

### Weekdays vs. weekends

In [ ]:
trips_silver.groupBy(
    "city", "is_weekend", "local_hour"
).count().orderBy(
    "city", "is_weekend", "local_hour"
).show(144)

In [ ]:
hourly_trips = trips_silver.groupBy(
    "city", "is_weekend", "local_hour"
).count()

window = Window.partitionBy("city", "is_weekend")

hourly_trips = hourly_trips.withColumn(
    "percentage",
    F.round(
        F.col("count") /
        F.sum("count"),over(Window) * 100,
        2
    )
)

hourly_trips.orderBy(
    "city", "is_weekend", "local_hour"
).show(144)

In [ ]:
hourly_pd = hourly_trips.toPandas()

for city in ["bergen", "oslo", "trondheim"]:
    city_data = hourly_pd[hourly_pd["city"] == city]

    plt.figure(figsize=(10, 4))

    for weekend in [False, True]:
        data = city_data[
            city_data["is_weekend"] == weekend
        ].sort_values("local_hour")

        plt.plot(
            data["local_hour"],
            data["percentage"],
            label="Weekend" if weekend else "Weekday"
        )

### Defining day parts

In [ ]:
trips_silver = trips_silver.withColumn(
    "day_part",
    F.when(
        F.col("local_hour").between(0, 5), "night"
    ).when(
        F.col("local_hour") == 6, "early_morning"
    ).when(
        F.col("local_hour").between(7, 9), "morning_rush"
    ).when(
        F.col("local_hour").between(10, 14), "midday"
    ).when(
        F.col("local_hour").between(15, 17), "athernoon_rush"
    ).when(
        F.col("local_hour").between(18, 23), "evening"
    )
)

## 8. Station Exploration

### Station identifiers

Something else that caught my eye is that the different cities share the same or similar station ID distributions, this may indicate that the different cities use the same identifications. So maybe the ID alone won't be possible to identify which city the trip was in.  

Let's investigate this a bit further:

In [ ]:
# Combine start and end station IDs
stations = trips_silver.select(
    "city",
    col("start_station_id").alias("station_id")
).union(
    trips_silver.select(
        "city",
        col("end_station_id").alias("station_id")
    )
).distinct()

# Find station IDs appearing in multiple cities
stations.groupBy("station_id").agg(
    countDistinct("city").alias("num_cities")
).filter(
    col("num_cities") > 1
).show()

Surpricingly enough, the investigation shows that no `start_station_id` or `end_station_id` is shared across cities in this current dataset. I do think including cities when grouping by station IDs to ensure robustness against potential overlaps in future data and it gives a better overview. 

It did however irritate me that I can't see a direct pattern between the cities and their stations, so let's see if there is one:

### Station ID distribution

In [ ]:
stations = (
    stations
    .select(
        "city",
        col("station_id").cast("int").alias("station_id")
    )
    .distinct()
)

for n in [2, 3, 4, 5, 10]:
    print(f"\nModulo {n}")

    stations.groupBy(
        "city",
        (col("station_id") % n).alias("remainder")
    ).agg(
        count("*").alias("num_stations")
    ).orderBy("city", "remainder").show()

The station IDs appear to be distributed without an obvious city-specific numerical pattern. Modulo analysis showed relatively balanced distributions across the three cities, with no clear preference for even or odd numbers or particular remainders. However, this does not establish whether IDs were assigned randomly or sequentially. 

### Station consistency

Next, we want to investigate whether the stations information associated with each ID remains consistent throughout the dataset. 

In [ ]:
start_stations = trips_silver.select(
    "city",
    F.col("start_station_id").alias("station_id"),
    F.col("start_station_name").alias("station_name"),
    F.col("start_station_latitude").alias("latitude"), 
    F.col("start_station_longitude").alias("longitude")
)

end_stations = trips_silver.select(
    "city", 
    F.col("end_station_id").alias("station_id"),
    F.col("end_station_name").alias("station_name"),
    F.col("end_station_latitude").alias("latitude"),
    F.col("end_station_longitude").alias("longitude")
)

stations = (
    start_stations
    .unionByName(end_stations)
    .distinct()
)

In [ ]:
station_consistency = (
    stations
    .groupBy("city", "station_id")
    .agg(
        F.countDistinct("station_name").alias("num_names"),
        F.countDistinct("latitude", "longitude").alias("num_locations")
    )
)

station_consistency.filter(
    (F.col("num_names") > 1) | (F.col("num_locations") > 1)
).show(50, truncate=False)

In [ ]:
inconsistent_stations = station_consistency.filter(
    (F.col("num_names") > 1) | (F.col("num_locations") > 1)
)

stations.join(
    inconsistent_stations,
    ["city", "station_id"],
    "inner"
).orderBy(
    "city", "station_id"
).show(50, truncate=False)

### Station coverage

In [ ]:
stations.groupBy("city").agg(
    F.countDistinct("station_id").alias("num_stations")
).show()

In [ ]:
monthly_start_station = trips_silver.select(
    "city",
    "year_month",
    F.col("start_station_id").alias("station_id")
)

monthly_end_station = trips_silver.select(
    "city",
    "year_month",
    F.col("end_station_id").alias("station_id")
)

monthly_stations = monthly_start_station.unionByName(monthly_end_station)

monthly_stations.groupBy(
    "city", "year_month"
).agg(
    F.countDistinct("station_id").alias("num_stations")
).orderBy(
    "city", "year_month"
).show(200)

### Station flow and imbalance

In [ ]:
departures = (
    trips_silver
    .groupBy("city", "start_station_id")
    .count()
    .withColumnRenamed("start_station_id", "station_id")
    .withColumnRenamed("count", "departures")
)

arrivals = (
    trips_silver
    .groupBy("city", "end_station_id")
    .count()
    .withColumnRenamed("end_station_id", "station_id")
    .withColumnRenamed("count", "arrivals")
)

station_flows = (
    departures
    .join(arrivals, ["city", "station_id"], "full")
    .fillna(0, subset=["departures", "arrivals"])
    .withColumn(
        "net_outflow",
        F.col("departures") - F.col("arrivals")
    )
)

In [ ]:
station_flows.orderBy(
    F.desc("net_outflow")
).show(20)

In [ ]:
station_flows.orderBy(
    F.asc("net_outflow")
).show(20)

## 9. Origin-Destination Analysis

### Which station pair are the most used?

In [ ]:
trips_silver.groupBy(
    "city",
    "start_station_id",
    "end_station_id"
).count().orderBy(
    col("count").desc()
).show(20, truncate=False)

In [ ]:
stations.groupBy("city").agg(
    F.countDistinct("station_id").alias("num_stations")
).show()

### Morning vs. athernoon routes

In [ ]:
rush_hour_routes = (
    trips_silver
    .filter(
        (~F.col("is_weekend")) &
        (~F.col("round_trip_flag")) &
        F.col("day_part").isin(
            "morning_rush", "afthernoon_rush"
        )
    ).groupBy(
        "city",
        "day_part",
        "start_station_id",
        "end_station_id"
    ).count()
)

In [ ]:
window = Window.partitionBy(
    "city", "day_part"
).orderBy(
    F.desc("count"),
    "start_station_id",
    "end_station_id"
)

rush_hour_routes.withColumn(
    "rank", F.row_number().over(window)
).filter(
    F.col("rank") <= 10
).orderBy(
    "city", "day_part", "rank"
).show(60)

## 10. Summary and Proposed Silver tranformations

| Category | Proposed transformations |
| ---------|--------------------------|
| Standardization | Data types, city, year_month, UTC, local timestamps |
| Data quality | Duplicates, duration validation, coordinate validation |
| Quality flags | Short trips, unrealistic speed, invalid duration, invalid coordinates |
| Temporal features | Local hour, weekday, weekend, hourly UTC keys |
| Trip characteristics | Haversin distance, minimum speed, round trip flag |
| Data modeling | Trip-level grain, composite station identifiers, potential station dimension |

## 11. Test silver_trips.py

In [ ]:
import sys
import os

print(os.getcwd())
sys.path.append("../Silver")

from silver_trips import (
    read_raw_trips,
    build_silver_trips,
    trips_quality_metrics
)

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

trips_silver_test = build_silver_trips(spark, RAW)

trips_silver_test.printSchema()
trips_silver_test.show(5, truncate=False)

In [ ]:
trips_quality_metrics(spark, RAW)